# 04. Digital Image Processing Change Detection Algorithms
### Project: Image Change Detection using Digital Image Processing for Environmental Monitoring

This notebook demonstrates:
1. Execution of 5 classical DIP change detection methods:
   - **Index Differencing**
   - **Image Differencing**
   - **PCA Differencing** (Principal Component Analysis on stacked spectral delta vectors)
   - **Change Vector Analysis** (CVA Euclidean magnitude)
   - **Structural Similarity** (1 - SSIM dissimilarity)
2. Thresholding strategies (**Otsu** vs **Adaptive Gaussian**).
3. Postprocessing: **Morphological Opening & Closing** + **Small region area filtering**.
4. Directional **Gain vs Loss** environmental classification.
5. Generating high-contrast **alpha-blended overlays**.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / "backend"))

from app.services.utils import load_image, generate_synthetic_pair, normalize_to_uint8
from app.services.preprocessing import preprocess_pair
from app.services.change_detection import run_method
from app.services.postprocessing import (
    morphological_cleanup,
    remove_small_regions,
    classify_gain_loss,
    make_overlay,
    compute_statistics
)


### Step 1: Prepare Preprocessed Image Pair


In [ ]:
project_root = Path.cwd().parent
pair_paths = generate_synthetic_pair("deforestation", size=512, base_dir=str(project_root))

before_info = load_image(pair_paths["before_path"])
after_info = load_image(pair_paths["after_path"])

clean_before, clean_after, valid_mask = preprocess_pair(
    before_info["bands"], after_info["bands"], do_denoise=True, do_match=True
)

print("Preprocessed pair ready.")


### Step 2: Compare Continuous Difference Maps Across All 5 Algorithms


In [ ]:
methods = ["index_differencing", "image_differencing", "pca", "cva", "ssim"]
results = {}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, m in enumerate(methods):
    res = run_method(m, clean_before, clean_after, use_case="deforestation", is_multispectral=True)
    results[m] = res
    im = axes[i].imshow(res["diff_map"], cmap="magma")
    axes[i].set_title(m.replace("_", " ").title(), fontsize=11, fontweight="bold")
    axes[i].axis("off")
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.suptitle("Comparison of Raw Difference Maps", fontsize=15, y=1.05)
plt.tight_layout()
plt.show()


### Step 3: Thresholding & Morphological Cleanup Pipeline
We demonstrate how morphological opening and small region filtering eliminate isolated false alarms.


In [ ]:
raw_mask = results["pca"]["mask"]
clean_morph = morphological_cleanup(raw_mask, ksize=5)
final_filtered = remove_small_regions(clean_morph, min_area=40)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(raw_mask, cmap="gray")
axes[0].set_title("1. Raw Otsu Segmented Mask", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(clean_morph, cmap="gray")
axes[1].set_title("2. Morphological Cleanup (Opening + Closing)", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(final_filtered, cmap="gray")
axes[2].set_title("3. Filtered Mask (Removed Regions < 40 px)", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()


### Step 4: Directional Gain vs Loss Classification & Quantitative Statistics


In [ ]:
gain_mask, loss_mask = classify_gain_loss(
    results["pca"]["signed_diff"],
    final_filtered,
    use_case="deforestation"
)

stats = compute_statistics(
    mask=final_filtered,
    signed_diff=results["pca"]["signed_diff"],
    pixel_area_m2=100.0,
    use_case="deforestation"
)

print("=" * 45)
print("STATISTICAL CHANGE SUMMARY")
print("=" * 45)
for k, v in stats.items():
    print(f"{k:<18}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(loss_mask, cmap="Reds")
axes[0].set_title(f"Vegetation Loss ({stats['loss_km2']} km^2)", fontweight="bold", color="darkred")
axes[0].axis("off")

axes[1].imshow(gain_mask, cmap="Greens")
axes[1].set_title(f"Vegetation Gain ({stats['gain_km2']} km^2)", fontweight="bold", color="darkgreen")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Step 5: Semi-Transparent Overlay Generation


In [ ]:
overlay_img = make_overlay(clean_after, final_filtered, color=(239, 68, 68), alpha=0.5)

plt.figure(figsize=(8, 8))
plt.imshow(overlay_img)
plt.title(f"Deforestation Change Overlay ({stats['percent_change']}% Changed Area)", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()
